# Fake News Detection (RoBERTa) + SHAP (XAI)

This notebook explains predictions from the fine-tuned checkpoint `hamzab/roberta-fake-news-classification` using SHAP token attributions.

Model input format (important for this checkpoint):
`<title> ... <content> ... <end>`

In the SHAP text plot (we explain `logit(P(FAKE))`):
- Red tokens increase `logit(P(FAKE))` (push toward the model's **FAKE** label)
- Blue tokens decrease it (push toward the opposite label, often **TRUE**)


In [ ]:
# If needed (fresh env / Colab):
# %pip install -q shap transformers torch matplotlib pandas


In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import shap
import matplotlib.pyplot as plt

def get_device() -> str:
    if torch.cuda.is_available():
        return 'cuda'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

device = get_device()
device


In [ ]:
MODEL_ID = 'hamzab/roberta-fake-news-classification'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.eval()
model.to(device)

def get_label_names(m) -> list[str]:
    id2label = getattr(m.config, 'id2label', None)
    if isinstance(id2label, dict) and id2label:
        try:
            items = sorted(((int(k), str(v)) for k, v in id2label.items()), key=lambda kv: kv[0])
            names = [v for _, v in items]
            if len(names) == int(getattr(m.config, 'num_labels', len(names))):
                return names
        except Exception:
            pass

    if int(getattr(m.config, 'num_labels', 2)) == 2:
        return ['Fake', 'Real']

    return [f'LABEL_{i}' for i in range(int(getattr(m.config, 'num_labels', 2)))]

label_names = get_label_names(model)
label_names


In [ ]:
def pick_fake_label(names: list[str]) -> str:
    # Handle common conventions: ['FAKE','TRUE'] or ['Fake','Real'], etc.
    if not names:
        return 'FAKE'

    for cand in ['FAKE', 'Fake', 'fake']:
        if cand in names:
            return cand

    for n in names:
        if 'fake' in n.lower():
            return n

    # Fallback: many fake-news checkpoints put FAKE at index 0
    return names[0]

fake_label = pick_fake_label(label_names)
fake_idx = label_names.index(fake_label) if label_names else 0
fake_label, fake_idx


In [ ]:
def format_input(title: str, content: str) -> str:
    # Required by hamzab/roberta-fake-news-classification model card
    return '<title>' + title + '<content>' + content + '<end>'

def _as_text_list(x):
    if isinstance(x, str):
        return [x]
    if isinstance(x, np.ndarray):
        return [str(v) for v in x.tolist()]
    return [str(v) for v in list(x)]

def predict_proba(texts, batch_size: int = 16, max_length: int = 512) -> np.ndarray:
    texts = _as_text_list(texts)
    if len(texts) == 0:
        return np.zeros((0, int(getattr(model.config, 'num_labels', 2))), dtype=np.float32)

    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        with torch.inference_mode():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()

        out.append(probs)

    return np.vstack(out)

def logit(p: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def predict_logit_fake(texts) -> np.ndarray:
    probs = predict_proba(texts)
    return logit(probs[:, fake_idx])


In [ ]:
import pandas as pd

# Option A (recommended): use the training distribution.
# Download the Kaggle "Fake and Real News" dataset (Fake.csv / True.csv)
# and point DATA_DIR to the folder containing those files.
DATA_DIR = Path('.')  # <-- change this
fake_csv = DATA_DIR / 'Fake.csv'
true_csv = DATA_DIR / 'True.csv'

def _safe_str(x) -> str:
    try:
        if pd.isna(x):
            return ''
    except Exception:
        pass
    return str(x)

other_label = next((n for n in label_names if n != fake_label), None)

examples_meta = []
if fake_csv.exists() and true_csv.exists():
    fake_df = pd.read_csv(fake_csv)
    true_df = pd.read_csv(true_csv)

    def pick_one(df: pd.DataFrame, want_pred: str, seed: int = 0, tries: int = 200):
        rng = np.random.default_rng(seed)
        if len(df) == 0:
            raise ValueError('Empty dataframe')
        for _ in range(tries):
            r = df.iloc[int(rng.integers(0, len(df)))]
            title = _safe_str(r.get('title', ''))
            content = _safe_str(r.get('text', r.get('content', '')))
            inp = format_input(title, content)
            p = predict_proba([inp])[0]
            pred = label_names[int(np.argmax(p))]
            if pred == want_pred:
                return title, content, p, pred
        # fallback if we can't find a matching prediction quickly
        r = df.sample(1, random_state=seed).iloc[0]
        title = _safe_str(r.get('title', ''))
        content = _safe_str(r.get('text', r.get('content', '')))
        inp = format_input(title, content)
        p = predict_proba([inp])[0]
        pred = label_names[int(np.argmax(p))]
        return title, content, p, pred

    # Try to select one example that the model predicts FAKE and one it predicts TRUE.
    t0, c0, _, _ = pick_one(fake_df, want_pred=fake_label, seed=0)
    t1, c1, _, _ = pick_one(true_df, want_pred=other_label or label_names[0], seed=1)

    examples_meta = [
        {'source': 'Fake.csv', 'title': t0, 'content': c0},
        {'source': 'True.csv', 'title': t1, 'content': c1},
    ]
else:
    print('Fake.csv/True.csv not found. Using toy examples; set DATA_DIR to use training-style samples.')
    examples_meta = [
        {
            'source': 'toy',
            'title': 'SHOCKING: Government confirms aliens live under the ocean',
            'content': 'A viral post claims a secret base was discovered. No evidence is provided.',
        },
        {
            'source': 'toy',
            'title': 'City council approves 2026 budget after late-night vote',
            'content': 'The budget includes funding for road repairs and a new public library branch.',
        },
    ]

examples = [format_input(e['title'], e['content']) for e in examples_meta]

probs = predict_proba(examples)
for i, p in enumerate(probs):
    row = {label_names[j]: float(p[j]) for j in range(len(label_names))}
    pred = label_names[int(np.argmax(p))]
    print(i, row, 'pred=', pred, 'source=', examples_meta[i].get('source'))

print(f'logit(P({fake_label})):', predict_logit_fake(examples))


In [ ]:
# SHAP explains a scalar output. We explain logit(P(FAKE)) so attributions add up cleanly.
# Note: some SHAP versions assert if output_names is a list for single-output models.
explainer = shap.Explainer(
    predict_logit_fake,
    tokenizer,
    algorithm='partition',
)

MAX_EVALS = 300  # higher => more accurate, slower
FIXED_CONTEXT = 1  # keeps word order context; helps coherent chunks


In [ ]:
# Local explanation for one example
sv_one = explainer([examples[0]], max_evals=MAX_EVALS, fixed_context=FIXED_CONTEXT)
shap.plots.text(sv_one[0])


In [ ]:
# Explanations for multiple examples
shap_values = explainer(examples, max_evals=MAX_EVALS, fixed_context=FIXED_CONTEXT)
shap.plots.text(shap_values)


In [ ]:
# The <title>/<content>/<end> delimiters are required by this checkpoint,
# but they are not part of the *news content*.
# When SHAP masks tokens it may assign large attributions to these delimiters
# simply because removing them changes the model's behavior.
#
# For XAI, it is often cleaner to treat delimiters as fixed formatting and fold
# their attributions into the base value (so the remaining tokens still add up).
def strip_delimiter_tokens(exp: shap.Explanation) -> shap.Explanation:
    toks = [str(t) for t in list(exp.data)]
    vals = np.asarray(exp.values, dtype=float)
    if vals.ndim > 1:
        vals = vals[:, 0]
    base = float(np.asarray(exp.base_values).reshape(-1)[0])

    tag_words = {'title', 'content', 'end'}
    drop = np.zeros(len(toks), dtype=bool)

    for i, tok in enumerate(toks):
        t = tok.strip()
        if t == '':
            drop[i] = True
            continue

        low = t.lower()
        prev = toks[i - 1] if i > 0 else ''
        nxt = toks[i + 1] if i + 1 < len(toks) else ''

        # Drop standalone angle bracket tokens that are adjacent to tag words
        if t in {'<', '>'} and (prev.strip().lower() in tag_words or nxt.strip().lower() in tag_words):
            drop[i] = True
            continue

        # Drop tag word tokens when bracket neighbors indicate they are delimiters
        if low in tag_words and ('<' in prev or '>' in nxt or prev.strip() == '<' or nxt.strip() == '>'):
            drop[i] = True
            continue

    dropped_sum = float(vals[drop].sum())
    keep_toks = []
    keep_vals = []
    for i, (tok, v) in enumerate(zip(toks, vals)):
        if drop[i]:
            continue
        # Remove any leftover bracket characters that got merged with punctuation (e.g. '.<' -> '.')
        cleaned = tok.replace('<', '').replace('>', '')
        if cleaned.strip() == '':
            dropped_sum += float(v)
            continue
        keep_toks.append(cleaned)
        keep_vals.append(float(v))

    return shap.Explanation(values=np.asarray(keep_vals), base_values=base + dropped_sum, data=keep_toks)

clean_shap_values = [strip_delimiter_tokens(shap_values[i]) for i in range(len(examples))]

# Clean visualization (delimiter tags removed):
for i in range(len(examples)):
    print('clean plot i=', i, 'pred=', label_names[int(np.argmax(probs[i]))], 'source=', examples_meta[i].get('source'))
    shap.plots.text(clean_shap_values[i])


In [ ]:
# Global token importance over the current set of examples (delimiter tags removed)
from collections import defaultdict

token_importance = defaultdict(float)
for sv in clean_shap_values:
    for tok, v in zip(sv.data, sv.values):
        t = str(tok).strip()
        if t == '':
            continue
        token_importance[t] += abs(float(v))

top = sorted(token_importance.items(), key=lambda kv: kv[1], reverse=True)[:20]
tokens = [t for t, _ in top][::-1]
imps = [v for _, v in top][::-1]

plt.figure(figsize=(9, 6))
plt.barh(tokens, imps)
plt.title(f'Global token importance (abs SHAP) for logit(P({fake_label}))')
plt.tight_layout()
plt.show()


In [ ]:
# Optional: export per-example interactive HTML + predictions
out_dir = Path('shap_out')
(out_dir / 'html').mkdir(parents=True, exist_ok=True)

with (out_dir / 'predictions.jsonl').open('w', encoding='utf-8') as f:
    for i, p in enumerate(probs):
        f.write(
            json.dumps(
                {
                    'i': i,
                    'labels': label_names,
                    'probs': {label_names[j]: float(p[j]) for j in range(len(label_names))},
                    'pred_label': label_names[int(np.argmax(p))],
                },
                ensure_ascii=True,
            )
            + '\n'
        )

for i in range(len(examples)):
    # Export the cleaned view (delimiter tags removed)
    html_obj = shap.plots.text(clean_shap_values[i], display=False)
    html_str = html_obj.data if hasattr(html_obj, 'data') else str(html_obj)
    full = (
        "<!doctype html><html><head><meta charset='utf-8'>"
        "<title>SHAP Text Explanation</title></head><body>"
        + html_str
        + "</body></html>"
    )
    (out_dir / 'html' / f'explain_{i:04d}.html').write_text(full, encoding='utf-8')

sorted((out_dir / 'html').glob('*.html'))


## Notes / troubleshooting

- If explanations are slow: reduce `MAX_EVALS`, shorten the text, and/or run on GPU (`cuda`).
- This checkpoint expects the `<title>...<content>...<end>` tags; missing tags can hurt both accuracy and explanation quality.
- For long documents, SHAP text explanations are approximate and may merge tokens into chunks (especially with `algorithm='partition'`).
